# Member 3: Image Caption Generation
## Baseline vs RAG (Group 11 - RAG Reranking)

**Author:** Member 3  
**Project:** Comparative study of RAG techniques for image captioning (MS COCO)  

---

### 📋 Pipeline Overview
1. Environment Setup
2. Imports
3. Clone GitHub Repo
4. Load BLIP-2 (Vision-Language Model)
5. Load CLIP (for retrieval)
6. Load FAISS index + captions from Member 2
7. Load Cross-Encoder Reranker
8. Define functions
9. Quick test on 1 image
10. Generate captions for 100 test images
11. Download results CSV

---
## 1. Environment Setup
Install all required libraries.

In [ ]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu pillow pandas tqdm

---
## 2. Imports
All Python libraries used in the pipeline.

In [ ]:
import os
import json
import subprocess
import urllib.request
import torch
import numpy as np
import pandas as pd
import faiss

from PIL import Image
from pathlib import Path
from tqdm import tqdm
from transformers import (
    Blip2Processor,
    Blip2ForConditionalGeneration,
    CLIPProcessor,
    CLIPModel
)
from sentence_transformers import CrossEncoder

print("✅ All libraries imported!")

---
## 3. Clone GitHub Repo
Download the shared repository containing Member 1's data and Member 2's FAISS index.

In [ ]:
REPO_URL = "https://github.com/Rachaaabn/ImageCaptioning-RagReranking.git"
REPO_DIR = Path("/content/repo")

if not REPO_DIR.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Pulling latest changes...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

print(f"✅ Repo ready at {REPO_DIR}")

---
## 4. Load BLIP-2
Vision-Language Model for caption generation. This is a 16 GB model, may take 3-5 minutes.

In [ ]:
print("Loading BLIP-2 (this may take a few minutes)...")
model_name = "Salesforce/blip2-flan-t5-xl"
processor = Blip2Processor.from_pretrained(model_name)
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
blip_model.eval()
print("✅ BLIP-2 loaded!")

---
## 5. Load CLIP
Used to convert test images into vectors that match Member 2's FAISS index (512-dim).

In [ ]:
print("Loading CLIP (for retrieval)...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("✅ CLIP loaded!")

---
## 6. Load FAISS Index + Captions
Member 2's retriever and the human captions used for RAG examples.

In [ ]:
# Load FAISS index
index = faiss.read_index(str(REPO_DIR / "outputs" / "faiss_index.bin"))
print(f"✅ FAISS index loaded ({index.ntotal} vectors)")

# Load valid IDs
with open(REPO_DIR / "outputs" / "valid_kb_ids.json", "r") as f:
    valid_kb_ids = json.load(f)

# Load captions for lookup
with open(REPO_DIR / "outputs" / "captions.json", "r") as f:
    all_captions = json.load(f)

caption_lookup = {}
for item in all_captions:
    img_id = item["image_id"]
    if img_id not in caption_lookup:
        caption_lookup[img_id] = []
    caption_lookup[img_id].append(item["caption_original"])

print(f"✅ Loaded {len(caption_lookup)} image captions for lookup")

---
## 7. Load Cross-Encoder Reranker
This is our **Group 11 specific technique**: it reranks the top-20 FAISS results to keep the best 5.

In [ ]:
print("Loading Cross-Encoder Reranker...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("✅ Reranker loaded!")

---
## 8. Define Functions
Three core functions:
- `get_rag_examples`: FAISS (20) + Reranker (5)
- `generate_baseline_caption`: BLIP-2 without examples
- `generate_rag_caption`: BLIP-2 with top-5 examples

In [ ]:
def get_rag_examples(test_image_path, top_k=5):
    """Retrieve Top-5 similar captions using FAISS + Reranker."""
    image = Image.open(test_image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")

    with torch.no_grad():
        features = clip_model.get_image_features(**inputs).pooler_output
        query_vec = features.numpy().flatten()
        query_vec = query_vec / np.linalg.norm(query_vec)
        query_vec = np.array([query_vec]).astype("float32")

    # FAISS: Top 20 candidates
    distances, indices = index.search(query_vec, 20)

    candidates = []
    for idx in indices[0]:
        kb_id = valid_kb_ids[idx]
        if kb_id in caption_lookup:
            candidates.append(caption_lookup[kb_id][0])

    # Reranker: keep Top 5
    query_text = "Describe this image"
    pairs = [[query_text, c] for c in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, candidates), reverse=True)
    return [text for score, text in ranked[:top_k]]


def generate_baseline_caption(image_path):
    """Generate caption without RAG."""
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(blip_model.device)
    with torch.no_grad():
        generated_ids = blip_model.generate(**inputs, max_new_tokens=50)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]


def generate_rag_caption(image_path):
    """Generate caption using Top-5 RAG examples."""
    examples = get_rag_examples(image_path, top_k=5)
    examples_text = "\n".join([f"- {ex}" for ex in examples])
    prompt = (
        f"Here are some examples of similar images:\n{examples_text}\n\n"
        "Now, describe the new image based on the visual details, "
        "using the style of the examples above."
    )

    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(blip_model.device)
    with torch.no_grad():
        generated_ids = blip_model.generate(**inputs, max_new_tokens=50)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]


print("✅ All functions defined!")

---
## 9. Quick Test on 1 Image
Test the pipeline on the first test image to verify everything works.

In [ ]:
# Load test IDs
test_ids = json.load(open(REPO_DIR / "outputs" / "splits.json"))["test_ids"]
os.makedirs("/content/test_images", exist_ok=True)

# Download first test image
first_id = test_ids[0]
img_str = str(first_id).zfill(12)
filepath = f"/content/test_images/{img_str}.jpg"

if not os.path.exists(filepath):
    urllib.request.urlretrieve(f"http://images.cocodataset.org/val2017/{img_str}.jpg", filepath)

print(f"Image: {filepath}")
print(f"Baseline: {generate_baseline_caption(filepath)}")
print(f"RAG:      {generate_rag_caption(filepath)}")

---
## 10. Generate Captions for 100 Test Images
This is the main experiment. It takes ~10-15 minutes on T4 GPU.
We generate **both** Baseline and RAG captions for each image.

In [ ]:
results = []

for test_id in tqdm(test_ids, desc="Processing test images"):
    img_str = str(test_id).zfill(12)
    filepath = f"/content/test_images/{img_str}.jpg"

    if not os.path.exists(filepath):
        try:
            urllib.request.urlretrieve(
                f"http://images.cocodataset.org/val2017/{img_str}.jpg",
                filepath
            )
        except Exception as e:
            print(f"Failed: {img_str} ({e})")
            continue

    try:
        baseline = generate_baseline_caption(filepath)
    except Exception as e:
        baseline = f"ERROR: {e}"

    try:
        rag = generate_rag_caption(filepath)
    except Exception as e:
        rag = f"ERROR: {e}"

    results.append({
        "image_id": test_id,
        "file_name": f"{img_str}.jpg",
        "baseline_caption": baseline,
        "rag_caption": rag
    })

df = pd.DataFrame(results)
df.to_csv("/content/rag_captions_comparison.csv", index=False)
print(f"\n✅ Saved {len(df)} captions!")
display(df.head())

---
## 11. Download Results CSV
Send the final CSV to your computer to upload to GitHub.

In [ ]:
from google.colab import files
files.download("/content/rag_captions_comparison.csv")